In [ ]:
import sys

%pip install pytorch-lightning wandb jiwer transformers torchaudio soundfile librosa huggingface-hub safetensors torchcodec
%pip install "focalcodec@git+https://github.com/lucadellalib/focalcodec.git@main#egg=focalcodec"

import importlib
for pkg in ["pytorch_lightning", "wandb", "jiwer", "transformers", "torchaudio", "soundfile",
            "librosa", "huggingface_hub", "safetensors", "torchcodec", "focalcodec"]:
    try:
        importlib.import_module(pkg)
        print(f"  ok  {pkg}")
    except ImportError as e:
        print(f" FAIL {pkg}: {e}")

In [ ]:
import os
import math
import warnings
from dataclasses import dataclass, field
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchaudio
import soundfile as sf
import librosa
import matplotlib.pyplot as plt
from tqdm import tqdm  # plain, not tqdm.auto -- see heartbeat cell below for why

import pytorch_lightning as pl
from torch.utils.data import Dataset, DataLoader, Sampler
from pytorch_lightning.loggers import WandbLogger
from pytorch_lightning.callbacks import ModelCheckpoint, EarlyStopping

import wandb
import jiwer
from transformers import GPT2TokenizerFast, GPT2LMHeadModel, pipeline as hf_pipeline
from focalcodec import FocalCodec

warnings.filterwarnings("ignore")
pl.seed_everything(42, workers=True)

In [ ]:
import threading
import time as _time

def _heartbeat(interval=2.0):
    while True:
        _time.sleep(interval)
        print(".", end="", flush=True)

threading.Thread(target=_heartbeat, args=(2.0,), daemon=True).start()
print("heartbeat thread started -- keeps Kaggle/papermill's IOPub watchdog satisfied during long silent computations")

In [ ]:
@dataclass
class Config:
    # data
    data_dir: Path = Path('/kaggle/input/datasets/mathurinache/the-lj-speech-dataset')
    splits_dir: Path = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path(".")
    max_total_len: int = 384
    debug_subset: int = None          # e.g. 64 for a fast smoke test end-to-end; None = full data

    # dataloader
    batch_size: int = 8
    num_workers: int = 2

    # optimization
    lr: float = 3e-4
    weight_decay: float = 0.01
    warmup_steps: int = 500
    max_epochs: int = 20
    accumulate_grad_batches: int = 4
    val_check_interval: float = 0.5
    early_stop_patience_epochs: int = 3   # stop if val/loss hasn't improved for this many epochs
    early_stop_min_delta: float = 1e-3    # ignore improvements smaller than this (avoid noise-triggered stops)
    precision: str = "16-mixed"       # T4 / P100 on Kaggle: fp16, not bf16 (no native bf16 tensor cores)

    # model
    base_model: str = "gpt2"
    codebook_size: int = 8192         # overwritten once the codec is loaded (cell below)

    # wandb
    wandb_project: str = "audioml-lab4-gpt-tts"
    wandb_run_name: str = None

    # test-time eval
    n_test_per_domain: int = 100      # cap per domain for CER/UTMOS+audio (None = full test set)

    @property
    def wavs_dir(self) -> Path:
        return self.data_dir / "wavs"

    @property
    def cache_dir(self) -> Path:
        d = self.splits_dir / "tokenized"; d.mkdir(parents=True, exist_ok=True); return d

    @property
    def ckpt_dir(self) -> Path:
        d = self.splits_dir / "checkpoints"; d.mkdir(parents=True, exist_ok=True); return d

    @property
    def test_audio_dir(self) -> Path:
        d = self.splits_dir / "test_audio"; d.mkdir(parents=True, exist_ok=True); return d


cfg = Config()
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"device={device}")
print(cfg)

In [ ]:

from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
secret_value_0 = user_secrets.get_secret("WANDB_API_KEY")
wandb.login(key=secret_value_0)


In [ ]:
train_df = pd.read_csv(cfg.splits_dir / "train.csv")
val_df   = pd.read_csv(cfg.splits_dir / "val.csv")
test_df  = pd.read_csv(cfg.splits_dir / "test.csv")

def attach_filepath(df):
    df = df.copy()
    df['filepath'] = df['id'].apply(lambda i: str(cfg.wavs_dir / f"{i}.wav"))
    return df

train_df, val_df, test_df = attach_filepath(train_df), attach_filepath(val_df), attach_filepath(test_df)

if cfg.debug_subset:
    train_df = train_df.iloc[:cfg.debug_subset].reset_index(drop=True)
    val_df   = val_df.iloc[:max(8, cfg.debug_subset // 4)].reset_index(drop=True)

print(f"train={len(train_df)}  val={len(val_df)}  test={len(test_df)}")
print(f"test domains: {test_df['domain'].value_counts().to_dict()}")

In [ ]:
codec = FocalCodec.from_pretrained('lucadellalib/focalcodec_25hz').to(device).eval()
for p in codec.parameters():
    p.requires_grad_(False)

CODEBOOK_SIZE = int(codec.codebook.shape[0])
CODEC_SR_IN = codec.sample_rate_input
CODEC_SR_OUT = codec.sample_rate_output
cfg.codebook_size = CODEBOOK_SIZE
print(f'codebook={CODEBOOK_SIZE}, sr_in/out={CODEC_SR_IN}/{CODEC_SR_OUT}')

def load_wav_16k(path):
    arr, sr = sf.read(path, dtype='float32', always_2d=False)
    if arr.ndim > 1:
        arr = arr.mean(-1)
    wav = torch.from_numpy(arr)
    if sr != CODEC_SR_IN:
        wav = torchaudio.functional.resample(wav, sr, CODEC_SR_IN)
    return wav  # (T,)

In [ ]:
tokenizer = GPT2TokenizerFast.from_pretrained(cfg.base_model)

@torch.no_grad()
def encode_split(df, cache_path, text_col='normalized_transcription', batch_size=16, overwrite=False):
    cache_path = Path(cache_path)
    if cache_path.exists() and not overwrite:
        print(f'load cache: {cache_path}')
        return torch.load(cache_path, weights_only=False)

    recs = df.to_dict('records')
    paths = [r['filepath'] for r in recs]
    order = np.argsort([os.path.getsize(p) for p in paths])  # similar-size wavs together -> less pad waste while encoding
    out = [None] * len(recs)
    for start in tqdm(range(0, len(recs), batch_size), desc=f'encode {cache_path.name}'):
        idxs = order[start:start + batch_size]
        wavs = [load_wav_16k(paths[i]) for i in idxs]
        lens = torch.tensor([w.numel() for w in wavs], dtype=torch.float32)
        L = int(lens.max())
        batch = torch.zeros(len(wavs), L)
        for j, w in enumerate(wavs):
            batch[j, :w.numel()] = w
        batch = batch.to(device)
        toks = codec.sig_to_toks(batch, length=(lens / L).to(device))
        tok_lens = ((lens / L).to(device) * toks.shape[-1]).round().clamp(max=toks.shape[-1]).to(torch.long)
        for j, i in enumerate(idxs):
            text_ids = tokenizer.encode(str(recs[i][text_col]), add_special_tokens=False)
            text_ids = torch.tensor(text_ids, dtype=torch.long)
            audio_ids = toks[j, :tok_lens[j]].to(torch.long).cpu()
            out[i] = {
                'id': recs[i]['id'],
                'text': recs[i][text_col],
                'text_ids': text_ids,
                'audio_ids': audio_ids,
                'n_text': int(text_ids.numel()),
                'n_audio': int(audio_ids.numel()),
            }

    cache_path.parent.mkdir(parents=True, exist_ok=True)
    torch.save(out, cache_path)
    print(f'saved {len(out)} items -> {cache_path}')
    return out

In [ ]:
train_items = encode_split(train_df, cfg.cache_dir / 'train.pt')
val_items   = encode_split(val_df,   cfg.cache_dir / 'val.pt')
test_items  = encode_split(test_df,  cfg.cache_dir / 'test.pt')

for name, items in [('train', train_items), ('val', val_items), ('test', test_items)]:
    a = np.array([r['n_audio'] for r in items]); t = np.array([r['n_text'] for r in items])
    print(f'{name:5s}: n={len(items):5d}  audio(mean/p95/max)={a.mean():.0f}/{np.percentile(a,95):.0f}/{a.max()}  '
          f'text(mean/p95/max)={t.mean():.0f}/{np.percentile(t,95):.0f}/{t.max()}')

In [ ]:
class TokenizedLJSpeech(Dataset):
    def __init__(self, items_or_path, max_total_len=384):
        items = torch.load(items_or_path, weights_only=False) if isinstance(items_or_path, (str, Path)) else items_or_path
        self.items = [it for it in items if it['n_text'] + it['n_audio'] + 2 <= max_total_len]
        dropped = len(items) - len(self.items)
        if dropped:
            print(f'dropped {dropped}/{len(items)} rows longer than max_total_len={max_total_len}')

    def __len__(self):
        return len(self.items)

    def __getitem__(self, i):
        it = self.items[i]
        return {'id': it['id'], 'text_ids': it['text_ids'].long(), 'audio_ids': it['audio_ids'].long()}


def collate(batch):
    B = len(batch)
    text_lens = torch.tensor([b['text_ids'].numel() for b in batch], dtype=torch.long)
    audio_lens = torch.tensor([b['audio_ids'].numel() for b in batch], dtype=torch.long)
    text_ids = torch.zeros(B, int(text_lens.max()), dtype=torch.long)
    audio_ids = torch.zeros(B, int(audio_lens.max()), dtype=torch.long)
    for i, b in enumerate(batch):
        text_ids[i, :b['text_ids'].numel()] = b['text_ids']
        audio_ids[i, :b['audio_ids'].numel()] = b['audio_ids']
    return {'text_ids': text_ids, 'text_lens': text_lens, 'audio_ids': audio_ids, 'audio_lens': audio_lens}


class LengthGroupedSampler(Sampler):
    """Shuffle, sort into length-similar chunks of size batch_size, then shuffle chunk order."""

    def __init__(self, lengths, batch_size, shuffle=True, seed=42):
        self.lengths = lengths
        self.batch_size = batch_size
        self.shuffle = shuffle
        self.seed = seed
        self.epoch = 0

    def __iter__(self):
        idx = list(range(len(self.lengths)))
        g = np.random.default_rng(self.seed + self.epoch)
        if self.shuffle:
            g.shuffle(idx)
        idx.sort(key=lambda i: self.lengths[i])  # stable sort: shuffle noise stays inside same-length buckets
        batches = [idx[i:i + self.batch_size] for i in range(0, len(idx), self.batch_size)]
        if self.shuffle:
            g.shuffle(batches)
        self.epoch += 1
        return iter([i for b in batches for i in b])

    def __len__(self):
        return len(self.lengths)

In [ ]:
@dataclass
class GPT2TTSConfig:
    base_model: str = "gpt2"
    codebook_size: int = 8192
    n_special_tokens: int = 2  # BOS, EOS

    @property
    def audio_vocab_size(self) -> int:
        return self.codebook_size + self.n_special_tokens

    @property
    def bos_id(self) -> int:
        return self.codebook_size

    @property
    def eos_id(self) -> int:
        return self.codebook_size + 1


class GPT2TTS(nn.Module):
    def __init__(self, cfg: GPT2TTSConfig):
        super().__init__()
        self.cfg = cfg
        self.base = GPT2LMHeadModel.from_pretrained(cfg.base_model)
        H = self.base.config.n_embd
        V = cfg.audio_vocab_size

        self.audio_emb = nn.Embedding(V, H)
        self.audio_head = nn.Linear(H, V, bias=False)
        nn.init.normal_(self.audio_emb.weight, std=0.02)
        nn.init.normal_(self.audio_head.weight, std=0.02)

    def _build_inputs(self, text_ids, text_lens, audio_ids, audio_lens):
        """Pack a variable-length batch into (B, L, H) embeddings + mask + labels.

        Per-row layout (tl = text_lens[i], al = audio_lens[i]):
            position :  0 .. tl-1   tl     tl+1 .. tl+al    tl+1+al    tl+2+al .. L-1
            content  :   text       BOS         audio          EOS          PAD
        """
        B, H = text_ids.size(0), self.base.config.n_embd
        device = text_ids.device
        lens = text_lens + audio_lens + 2
        L = int(lens.max())

        inputs = torch.zeros(B, L, H, device=device, dtype=self.audio_emb.weight.dtype)
        mask = torch.zeros(B, L, dtype=torch.long, device=device)
        labels = torch.full((B, L), -100, dtype=torch.long, device=device)

        wte = self.base.transformer.wte
        bos_emb = self.audio_emb.weight[self.cfg.bos_id]
        eos_emb = self.audio_emb.weight[self.cfg.eos_id]

        for i in range(B):
            tl, al = int(text_lens[i]), int(audio_lens[i])
            t_emb = wte(text_ids[i, :tl])
            a_ids = audio_ids[i, :al]
            seq = torch.cat([t_emb, bos_emb[None], self.audio_emb(a_ids), eos_emb[None]], dim=0)
            n = seq.size(0)

            inputs[i, :n] = seq
            mask[i, :n] = 1
            labels[i, tl + 1: tl + 1 + al] = a_ids
            labels[i, tl + 1 + al] = self.cfg.eos_id

        return inputs, mask, labels

    def forward(self, text_ids, text_lens, audio_ids, audio_lens):
        inputs, mask, labels = self._build_inputs(text_ids, text_lens, audio_ids, audio_lens)
        h = self.base.transformer(inputs_embeds=inputs, attention_mask=mask, return_dict=True).last_hidden_state
        logits = self.audio_head(h).float()  # (B, L, V), fp32 for stable softmax/CE under fp16 autocast

        shift_logits = logits[:, :-1, :].contiguous()
        shift_labels = labels[:, 1:].contiguous()
        loss = F.cross_entropy(
            shift_logits.reshape(-1, shift_logits.size(-1)),
            shift_labels.reshape(-1),
            ignore_index=-100,
        )
        return {"loss": loss, "logits": logits, "labels": labels}

    @torch.no_grad()
    def generate_audio(self, text_ids, max_new_tokens=400, temperature=1.0, top_k=None, top_p=None,
                        repetition_penalty=1.0):
        """Autoregressive sampling conditioned on text, with a running KV cache.

        top_k=1 == greedy decoding. Set top_k / top_p / repetition_penalty independently to compare
        decoding strategies at test time.
        """
        self.eval()
        device = text_ids.device
        text_ids = text_ids.view(-1).long().to(device)
        text_emb = self.base.transformer.wte(text_ids).unsqueeze(0)
        bos = self.audio_emb.weight[self.cfg.bos_id].view(1, 1, -1)
        prefix = torch.cat([text_emb, bos], dim=1)
        mask = torch.ones(1, prefix.size(1), device=device, dtype=torch.long)

        out = self.base.transformer(inputs_embeds=prefix, attention_mask=mask, use_cache=True, return_dict=True)
        past = out.past_key_values
        h = out.last_hidden_state[:, -1, :]

        generated = []
        for _ in range(max_new_tokens):
            logits = self.audio_head(h).float()  # (1, V)

            if repetition_penalty != 1.0 and generated:
                for tok in set(generated):
                    logits[0, tok] = logits[0, tok] / repetition_penalty if logits[0, tok] > 0 else logits[0, tok] * repetition_penalty

            logits = logits / max(temperature, 1e-5)

            if top_k is not None:
                v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                logits[logits < v[:, [-1]]] = -float('inf')

            if top_p is not None:
                sorted_logits, sorted_idx = torch.sort(logits, descending=True, dim=-1)
                probs_sorted = F.softmax(sorted_logits, dim=-1)
                cum_probs = probs_sorted.cumsum(dim=-1)
                cutoff = int((cum_probs > top_p).float().argmax(dim=-1).item())
                keep = sorted_idx[:, :cutoff + 1]
                filtered = torch.full_like(logits, float('-inf'))
                filtered.scatter_(1, keep, logits.gather(1, keep))
                logits = filtered

            probs = F.softmax(logits, dim=-1)
            next_id = torch.multinomial(probs, 1)
            token = int(next_id.item())
            if token == self.cfg.eos_id:
                break
            generated.append(token)

            mask = torch.cat([mask, torch.ones(1, 1, device=device, dtype=torch.long)], dim=1)
            out = self.base.transformer(
                inputs_embeds=self.audio_emb(next_id).view(1, 1, -1),
                attention_mask=mask, past_key_values=past, use_cache=True, return_dict=True,
            )
            past = out.past_key_values
            h = out.last_hidden_state[:, -1, :]

        return torch.tensor(generated, dtype=torch.long, device=device)

    @torch.no_grad()
    def generate_audio_beam(self, text_ids, beam_size=4, max_new_tokens=400, length_penalty=1.0):
        """Beam search over the codec vocabulary.

        Keeps `beam_size` hypotheses alive in a single batched forward pass. A finished beam
        (already emitted EOS) is frozen: its only allowed continuation is a zero-cost EOS no-op, so
        it keeps competing on score without extending further. Finished hypotheses are also copied
        into `finished` as soon as they complete -- a beam that falls out of the live top-`beam_size`
        pool in a later step must not be lost, since it may still be the best *completed* hypothesis
        overall. EOS itself is never appended to the returned sequence (mirrors `generate_audio`,
        and keeps the output valid input for `codec.toks_to_sig`, which only knows the real
        `0..codebook_size-1` codes). KV cache / attention mask are reordered by `beam_idx` every
        step, since beam k's parent at step t is not necessarily beam k's own previous state.
        """
        self.eval()
        device = text_ids.device
        beam_size = min(beam_size, self.cfg.audio_vocab_size)
        text_ids = text_ids.view(-1).long().to(device)
        text_emb = self.base.transformer.wte(text_ids).unsqueeze(0)         # (1, T, H)
        bos = self.audio_emb.weight[self.cfg.bos_id].view(1, 1, -1)
        prefix = torch.cat([text_emb, bos], dim=1)                          # (1, T+1, H)
        mask = torch.ones(1, prefix.size(1), device=device, dtype=torch.long)

        out = self.base.transformer(inputs_embeds=prefix, attention_mask=mask, use_cache=True, return_dict=True)
        past = out.past_key_values                                          # batch = 1
        h = out.last_hidden_state[:, -1, :]                                  # (1, H)

        # expand batch dim 1 -> beam_size
        past = tuple(tuple(t.repeat_interleave(beam_size, dim=0) for t in layer) for layer in past)
        mask = mask.repeat_interleave(beam_size, dim=0)                     # (beam_size, T+1)
        h = h.repeat_interleave(beam_size, dim=0)                            # (beam_size, H)

        V = self.cfg.audio_vocab_size
        beam_scores = torch.full((beam_size,), float('-inf'), device=device)
        beam_scores[0] = 0.0   # only beam 0 is "real" at step 0, rest are duplicates until they win on merit
        beam_seqs = [[] for _ in range(beam_size)]
        beam_done = [False] * beam_size
        finished = []   # list of (length_normalized_score, seq) -- completed hyps, kept even if pruned later

        for _ in range(max_new_tokens):
            logits = self.audio_head(h).float()                            # (beam_size, V)
            log_probs = F.log_softmax(logits, dim=-1)
            cand_scores = beam_scores.unsqueeze(1) + log_probs             # (beam_size, V)

            for i in range(beam_size):
                if beam_done[i]:
                    cand_scores[i] = float('-inf')
                    cand_scores[i, self.cfg.eos_id] = beam_scores[i]       # frozen no-op continuation

            flat = cand_scores.view(-1)
            top_scores, top_idx = flat.topk(beam_size)
            beam_idx = torch.div(top_idx, V, rounding_mode='floor')
            token_idx = top_idx % V

            new_seqs, new_done = [], []
            for k in range(beam_size):
                src, tok = int(beam_idx[k].item()), int(token_idx[k].item())
                is_eos = tok == self.cfg.eos_id
                seq = beam_seqs[src] + ([] if (beam_done[src] or is_eos) else [tok])  # EOS never enters the sequence
                done = beam_done[src] or is_eos
                if is_eos and not beam_done[src]:
                    norm = top_scores[k].item() / (max(len(seq), 1) ** length_penalty)
                    finished.append((norm, seq))
                new_seqs.append(seq)
                new_done.append(done)
            beam_seqs, beam_done, beam_scores = new_seqs, new_done, top_scores

            # reorder KV cache / mask to each survivor's parent, then advance one step
            past = tuple(tuple(t.index_select(0, beam_idx) for t in layer) for layer in past)
            mask = mask.index_select(0, beam_idx)
            mask = torch.cat([mask, torch.ones(beam_size, 1, device=device, dtype=torch.long)], dim=1)

            next_tokens = torch.tensor(
                [s[-1] if s else self.cfg.eos_id for s in beam_seqs], device=device,
            )
            out = self.base.transformer(
                inputs_embeds=self.audio_emb(next_tokens).view(beam_size, 1, -1),
                attention_mask=mask, past_key_values=past, use_cache=True, return_dict=True,
            )
            past = out.past_key_values
            h = out.last_hidden_state[:, -1, :]

            if all(beam_done):
                break

        if finished:
            best_seq = max(finished, key=lambda x: x[0])[1]
        else:
            candidates = [
                (score / (max(len(seq), 1) ** length_penalty), seq)
                for score, seq in zip(beam_scores.tolist(), beam_seqs)
            ]
            best_seq = max(candidates, key=lambda x: x[0])[1]
        return torch.tensor(best_seq, dtype=torch.long, device=device)


In [ ]:
class TTSDataModule(pl.LightningDataModule):
    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg

    def setup(self, stage=None):
        self.train_ds = TokenizedLJSpeech(cfg.cache_dir / 'train.pt', max_total_len=self.cfg.max_total_len)
        self.val_ds = TokenizedLJSpeech(cfg.cache_dir / 'val.pt', max_total_len=self.cfg.max_total_len)

    def train_dataloader(self):
        lengths = [it['n_text'] + it['n_audio'] for it in self.train_ds.items]
        sampler = LengthGroupedSampler(lengths, self.cfg.batch_size, shuffle=True)
        return DataLoader(self.train_ds, batch_size=self.cfg.batch_size, sampler=sampler,
                           collate_fn=collate, num_workers=self.cfg.num_workers, drop_last=True)

    def val_dataloader(self):
        return DataLoader(self.val_ds, batch_size=self.cfg.batch_size, shuffle=False,
                           collate_fn=collate, num_workers=self.cfg.num_workers)


class GPT2TTSLightning(pl.LightningModule):
    def __init__(self, model_cfg: GPT2TTSConfig, lr=3e-4, weight_decay=0.01, warmup_steps=500):
        super().__init__()
        self.save_hyperparameters()
        self.model = GPT2TTS(model_cfg)

    def _step(self, batch, stage):
        out = self.model(text_ids=batch['text_ids'], text_lens=batch['text_lens'],
                          audio_ids=batch['audio_ids'], audio_lens=batch['audio_lens'])
        loss, logits, labels = out['loss'], out['logits'], out['labels']

        with torch.no_grad():
            preds = logits[:, :-1, :].argmax(-1)
            tgt = labels[:, 1:]
            valid = tgt != -100
            acc = (preds[valid] == tgt[valid]).float().mean() if valid.any() else torch.tensor(0.0, device=self.device)

        bs = batch['text_ids'].size(0)
        self.log(f'{stage}/loss', loss, prog_bar=True, batch_size=bs, sync_dist=True)
        self.log(f'{stage}/ppl', torch.exp(loss.detach()), batch_size=bs, sync_dist=True)
        self.log(f'{stage}/token_acc', acc, prog_bar=True, batch_size=bs, sync_dist=True)
        return loss

    def training_step(self, batch, batch_idx):
        return self._step(batch, 'train')

    def validation_step(self, batch, batch_idx):
        self._step(batch, 'val')

    def configure_optimizers(self):
        opt = torch.optim.AdamW(self.parameters(), lr=self.hparams.lr, weight_decay=self.hparams.weight_decay)
        total_steps = max(1, self.trainer.estimated_stepping_batches)

        def lr_lambda(step):
            if step < self.hparams.warmup_steps:
                return step / max(1, self.hparams.warmup_steps)
            progress = (step - self.hparams.warmup_steps) / max(1, total_steps - self.hparams.warmup_steps)
            return 0.5 * (1 + math.cos(math.pi * min(max(progress, 0.0), 1.0)))

        sched = torch.optim.lr_scheduler.LambdaLR(opt, lr_lambda)
        return {"optimizer": opt, "lr_scheduler": {"scheduler": sched, "interval": "step"}}

## Periodic CER/UTMOS check during training (fixed subset, not continuous)

In [ ]:
FIXED_EVAL_DF = val_df.sample(n=min(40, len(val_df)), random_state=42).reset_index(drop=True)

asr_pipe = hf_pipeline(
    "automatic-speech-recognition", model="openai/whisper-base.en",
    device=0 if torch.cuda.is_available() else -1,
)
utmos_predictor = torch.hub.load("tarepan/SpeechMOS:v1.2.0", "utmos22_strong", trust_repo=True).to(device).eval()

def compute_cer(reference_text, wav, sr):
    if sr != 16000:
        wav = torchaudio.functional.resample(torch.from_numpy(wav), sr, 16000).numpy()
        sr = 16000
    hyp = asr_pipe({"array": wav, "sampling_rate": sr})["text"]
    return jiwer.cer(reference_text, hyp), hyp

def compute_utmos(wav, sr):
    wav_t = torch.from_numpy(wav).float().unsqueeze(0).to(device)
    with torch.no_grad():
        score = utmos_predictor(wav_t, sr)
    return float(score.item())


class PeriodicGenEvalCallback(pl.Callback):
    """CER/UTMOS on a small FIXED subset at ~25/50/75/100% of training EPOCHS -- not every epoch.

    Epoch-based rather than step-based: with `val_check_interval=0.5` you already validate twice
    per epoch, so this isn't coarser than what you're already looking at, and "epoch 5/10/15/20" is
    easier to talk about in a report than a raw step count.

    NOTE: this loop (40 rows x autoregressive generation + Whisper CER + UTMOS, no built-in output)
    was the actual cause of a `CellTimeoutError: Timeout waiting for IOPub` on Kaggle's "Save & Run
    All" (papermill) -- it fired right at epoch 4 (the first 25% trigger) with total silence the
    whole time. The background heartbeat thread (defined earlier) should prevent that regardless,
    but this loop also prints its own progress now, for both safety and visibility.
    """

    def __init__(self, fixed_df, fractions=(0.25, 0.5, 0.75, 1.0), gen_kwargs=None, log_n_audio=3):
        super().__init__()
        self.fixed_df = fixed_df
        self.fractions = fractions
        self.gen_kwargs = gen_kwargs or dict(temperature=0.9, top_k=100)
        self.log_n_audio = log_n_audio
        self._target_epochs = None
        self._done_epochs = set()

    def on_train_epoch_end(self, trainer, pl_module):
        if self._target_epochs is None:
            total = max(1, trainer.max_epochs)
            # current_epoch is 0-indexed (counts completed epochs); map each fraction to the epoch
            # INDEX whose completion corresponds to it, clamped so 100% always lands on the last epoch.
            self._target_epochs = sorted({
                max(0, min(int(round(total * f)) - 1, total - 1)) for f in self.fractions
            })

        epoch = trainer.current_epoch
        due = [e for e in self._target_epochs if epoch >= e and e not in self._done_epochs]
        if not due:
            return
        self._done_epochs.update(due)
        self._run_eval(trainer, pl_module, epoch)

    @torch.no_grad()
    def _run_eval(self, trainer, pl_module, epoch):
        model = pl_module.model
        was_training = model.training
        model.eval()

        cers, utmoses, audio_logs = [], [], []
        n_rows = len(self.fixed_df)
        print(f"[checkpoint_eval epoch={epoch}] scoring {n_rows} fixed rows (CER+UTMOS)...", flush=True)
        for i, row in self.fixed_df.iterrows():
            if i % 10 == 0:
                print(f"  [checkpoint_eval epoch={epoch}] {i}/{n_rows}", flush=True)
            text_ids = torch.tensor(
                tokenizer.encode(str(row['normalized_transcription']), add_special_tokens=False),
                device=pl_module.device,
            )
            audio_ids = model.generate_audio(text_ids, max_new_tokens=400, **self.gen_kwargs)
            if audio_ids.numel() == 0:
                continue
            wav = codec.toks_to_sig(audio_ids.unsqueeze(0)).squeeze(0).detach().cpu().numpy().astype(np.float32)
            cer, _ = compute_cer(row['normalized_transcription'], wav, CODEC_SR_OUT)
            utmos = compute_utmos(wav, CODEC_SR_OUT)
            cers.append(cer)
            utmoses.append(utmos)
            if i < self.log_n_audio:
                audio_logs.append(wandb.Audio(wav, sample_rate=CODEC_SR_OUT, caption=f"{row['id']} cer={cer:.2f}"))

        if was_training:
            model.train()

        pct = round(100 * (epoch + 1) / max(1, trainer.max_epochs))
        log_dict = {
            "checkpoint_eval/epoch": epoch,
            "checkpoint_eval/pct_complete": pct,
            "checkpoint_eval/cer": float(np.mean(cers)) if cers else float('nan'),
            "checkpoint_eval/utmos": float(np.mean(utmoses)) if utmoses else float('nan'),
            "checkpoint_eval/n_samples": len(cers),
        }
        if audio_logs:
            log_dict["checkpoint_eval/audio_samples"] = audio_logs
        trainer.logger.experiment.log(log_dict)
        print(f"[checkpoint_eval epoch={epoch} ({pct}%)] CER={log_dict['checkpoint_eval/cer']:.3f}  "
              f"UTMOS={log_dict['checkpoint_eval/utmos']:.3f}  n={len(cers)}")


periodic_eval_cb = PeriodicGenEvalCallback(FIXED_EVAL_DF)

In [ ]:
model_cfg = GPT2TTSConfig(base_model=cfg.base_model, codebook_size=cfg.codebook_size)
lit_model = GPT2TTSLightning(model_cfg, lr=cfg.lr, weight_decay=cfg.weight_decay, warmup_steps=cfg.warmup_steps)
datamodule = TTSDataModule(cfg)

wandb_logger = WandbLogger(project=cfg.wandb_project, name=cfg.wandb_run_name, log_model=False)
wandb_logger.log_hyperparams(vars(cfg))

checkpoint_cb = ModelCheckpoint(
    dirpath=cfg.ckpt_dir,
    filename='best-{epoch:02d}-{step}',
    monitor='val/token_acc',
    mode='max',
    save_top_k=1,
    save_last=True,      # always keeps checkpoints/last.ckpt up to date
    auto_insert_metric_name=False,
)
checks_per_epoch = max(1, round(1 / cfg.val_check_interval)) if cfg.val_check_interval <= 1 else 1
early_stop_patience = cfg.early_stop_patience_epochs * checks_per_epoch

early_stop_cb = EarlyStopping(
    monitor='val/loss',
    mode='min',                              # stop when val/loss stops decreasing --
    patience=early_stop_patience,            # covers both "plateaued" and "started rising" (overfitting):
    min_delta=cfg.early_stop_min_delta,       # either way, val/loss fails to hit a new minimum for N checks
    verbose=True,
)

class TrainHeartbeatCallback(pl.Callback):
    """Plain `print` every N steps -- real stdout, unlike Lightning's default (ipywidgets-based)
    progress bar, which is disabled below (`enable_progress_bar=False`) for the same papermill
    IOPub reason as the heartbeat thread above."""

    def __init__(self, every_n_steps=20):
        super().__init__()
        self.every_n_steps = every_n_steps

    def on_train_batch_end(self, trainer, pl_module, outputs, batch, batch_idx):
        step = trainer.global_step
        if step % self.every_n_steps == 0:
            loss = trainer.callback_metrics.get('train/loss')
            print(f"[train] epoch={trainer.current_epoch} step={step} "
                  f"train/loss={loss:.4f}" if loss is not None else
                  f"[train] epoch={trainer.current_epoch} step={step}", flush=True)


train_heartbeat_cb = TrainHeartbeatCallback(every_n_steps=20)

trainer = pl.Trainer(
    max_epochs=cfg.max_epochs,
    accelerator='gpu' if torch.cuda.is_available() else 'cpu',
    devices=1,
    precision=cfg.precision,
    logger=wandb_logger,
    callbacks=[checkpoint_cb, periodic_eval_cb, early_stop_cb, train_heartbeat_cb],
    enable_progress_bar=False,  # widget-based bar doesn't reliably satisfy papermill's IOPub watchdog
    gradient_clip_val=1.0,
    accumulate_grad_batches=cfg.accumulate_grad_batches,
    val_check_interval=cfg.val_check_interval,
    log_every_n_steps=20,
)

trainer.fit(lit_model, datamodule=datamodule)
print(f"best checkpoint: {checkpoint_cb.best_model_path} (val/token_acc={checkpoint_cb.best_model_score})")
print(f"last checkpoint: {checkpoint_cb.last_model_path}")
print(f"stopped at epoch {trainer.current_epoch} "
      f"(early_stop_cb.stopped_epoch={early_stop_cb.stopped_epoch or 'n/a -- ran to max_epochs'})")


## 10. Test evaluation — CER + UTMOS

In [ ]:
best_ckpt_path = checkpoint_cb.best_model_path or checkpoint_cb.last_model_path
print(f'loading {best_ckpt_path} for test eval')
eval_model = GPT2TTSLightning.load_from_checkpoint(best_ckpt_path, model_cfg=model_cfg).model.to(device).eval()

In [ ]:
SAMPLING_CONFIGS = {
    # "greedy": {"method": "sampling", "kwargs": dict(temperature=1.0, top_k=1, top_p=None, repetition_penalty=1.0)},
    "top_k":  {"method": "sampling", "kwargs": dict(temperature=0.9, top_k=100, top_p=None, repetition_penalty=1.0)},
    # "beam":   {"method": "beam",     "kwargs": dict(beam_size=4, length_penalty=1.0)},
}

def run_test_eval(model, test_df, sampling_name, sampling_cfg, out_dir, n_per_domain):
    model.eval()
    method, kwargs = sampling_cfg["method"], sampling_cfg["kwargs"]
    results = []
    for domain, group in test_df.groupby('domain'):
        rows = group if n_per_domain is None else group.sample(min(n_per_domain, len(group)), random_state=42)
        for _, row in tqdm(rows.iterrows(), total=len(rows), desc=f'{sampling_name}/{domain}'):
            text_ids = torch.tensor(tokenizer.encode(str(row['normalized_transcription']), add_special_tokens=False), device=device)
            if method == "beam":
                audio_ids = model.generate_audio_beam(text_ids, max_new_tokens=400, **kwargs)
            else:
                audio_ids = model.generate_audio(text_ids, max_new_tokens=400, **kwargs)
            if audio_ids.numel() == 0:
                continue
            with torch.no_grad():
                wav = codec.toks_to_sig(audio_ids.unsqueeze(0)).squeeze(0).detach().cpu().numpy().astype(np.float32)
            sr = CODEC_SR_OUT
            cer, hyp = compute_cer(row['normalized_transcription'], wav, sr)
            utmos = compute_utmos(wav, sr)

            wav_path = out_dir / sampling_name / domain / f"{row['id']}.wav"
            wav_path.parent.mkdir(parents=True, exist_ok=True)
            sf.write(wav_path, wav, sr)

            results.append({
                'id': row['id'], 'domain': domain, 'sampling': sampling_name,
                'text': row['normalized_transcription'], 'asr_hyp': hyp,
                'cer': cer, 'utmos': utmos, 'wav_path': str(wav_path), 'sr': sr,
            })
    return pd.DataFrame(results)


all_results = []
for name, sampling_cfg in SAMPLING_CONFIGS.items():
    df_res = run_test_eval(eval_model, test_df, name, sampling_cfg, cfg.test_audio_dir, cfg.n_test_per_domain)
    all_results.append(df_res)
results_df = pd.concat(all_results, ignore_index=True)
results_df.to_csv(cfg.test_audio_dir / "test_eval_results.csv", index=False)
print(f"{len(results_df)} generated+scored rows -> {cfg.test_audio_dir / 'test_eval_results.csv'}")
results_df.groupby(['sampling', 'domain'])[['cer', 'utmos']].mean()

In [ ]:
for (sampling, domain), grp in results_df.groupby(['sampling', 'domain']):
    wandb.log({
        f"test/{sampling}/{domain}/cer": grp['cer'].mean(),
        f"test/{sampling}/{domain}/utmos": grp['utmos'].mean(),
        f"test/{sampling}/{domain}/n_samples": len(grp),
    })

audio_table = wandb.Table(columns=["id", "domain", "sampling", "text", "asr_hyp", "cer", "utmos", "audio"])
for _, r in results_df.iterrows():
    audio_table.add_data(
        r['id'], r['domain'], r['sampling'], r['text'], r['asr_hyp'], r['cer'], r['utmos'],
        wandb.Audio(r['wav_path'], sample_rate=int(r['sr'])),
    )
wandb.log({"test/audio_samples": audio_table})

wandb.finish()
print(f"generated audio saved under: {cfg.test_audio_dir}")